# Define a custom discipline and couple it to electrical physics

We define a temperature-rise/heat-flow discipline with no registration, then connect a Joule heater to a thermal resistance and thermal capacitance. The zero thermal reference means ambient temperature **rise**, not absolute zero kelvin. The second example defines a velocity/force discipline.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from circulax import Nature, Discipline, TolerancePolicy, compile_circuit
from circulax.components.base_component import component
from circulax import electrical
from circulax.components.electronic import VoltageSource

temperature_rise = Nature("temperature_rise", units="K", atol=1e-5)
heat_rate = Nature("heat_rate", units="W", atol=1e-10)
heat = Discipline("heat", potential=temperature_rise, flow=heat_rate)

@component(ports=("p", "n", "hot", "cold"), discipline=electrical,
           port_disciplines={"hot": heat, "cold": heat})
def Heater(signals, resistance=100.0):
    voltage = signals.p - signals.n
    current = voltage / resistance
    power = voltage * current
    return {"p": current, "n": -current, "hot": -power, "cold": power}, {}

@component(ports=("hot", "cold"), discipline=heat)
def ThermalRC(signals, R_th=100.0, C_th=0.01):
    rise = signals.hot - signals.cold
    power = rise / R_th
    energy = C_th * rise
    return {"hot": power, "cold": -power}, {"hot": energy, "cold": -energy}


In [ ]:
net = {
    "instances": {"GND": {"component": "ground"}, "V1": {"component": "v", "settings": {"V": 2.0}},
                  "H1": {"component": "heater"}, "T1": {"component": "thermal_rc"}},
    "connections": {"V1,p1": "H1,p", "V1,p2": "GND,p1", "H1,n": "GND,p1",
                    "H1,hot": "T1,hot", "H1,cold": "GND,p1", "T1,cold": "GND,p1"},
}
models = {"ground": lambda: 0, "v": VoltageSource, "heater": Heater, "thermal_rc": ThermalRC}
circuit = compile_circuit(net, models, backend="dense", tolerances=TolerancePolicy())
y, converged = circuit.dc_checked()
assert converged
rise = y[circuit.port_map["H1,hot"]]
assert np.isclose(rise, 4.0, atol=1e-5)
print("Steady temperature rise [K]:", float(rise))
print("Analytic: (2 V)^2 / 100 ohm * 100 K/W = 4 K")
print("Equation error:", float(circuit.dc_diagnostics(y)["residual_error"]))


## One DAE contract across domains

The thermal storage dictionary contains energy, so its time derivative contributes power to the thermal balance. The same `(F, Q)` contract works for electrical charge, inductor flux, and mechanical momentum. No solver-specific heat-transfer code is needed.

In [ ]:
ts = jnp.linspace(0, 5, 101)
y0 = jnp.zeros(circuit.sys_size).at[circuit.port_map["V1,p1"]].set(2.0).at[circuit.port_map["V1,i_src"]].set(-0.02)
solution = circuit.transient(t0=0, t1=5, dt0=0.01, y0=y0, saveat=ts,
                             stepsize_controller=circuit.adaptive_controller(), throw=True, max_steps=20000)
transient_rise = solution.ys[:, circuit.port_map["H1,hot"]]
reference = 4 * (1 - np.exp(-np.asarray(ts)))  # R_th * C_th = 1 s
assert np.max(np.abs(transient_rise - reference)) < 0.02
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ts, transient_rise, label="Electrothermal DAE")
ax.plot(ts, reference, "--", label="Analytic")
ax.set(xlabel="Time [s]", ylabel="Temperature rise [K]")
ax.legend(); ax.grid(alpha=.3)
plt.show()


## Internal unknowns and equations are independent

For an inductor, `state_natures={"i": current}` and `equation_natures={"i": voltage}` declare a current unknown and a voltage residual. User models can choose arbitrary pairs; the compiler never infers units from the state name. Nature definitions are immutable and per-simulation overrides leave the model unchanged.

In [ ]:
velocity = Nature("velocity", units="m/s", atol=1e-8)
force = Nature("force", units="N", atol=1e-10)
mechanical = Discipline("mechanical", potential=velocity, flow=force)

@component(ports=("p", "n"), discipline=mechanical)
def Damper(signals, drag=2.0):
    force = drag * (signals.p - signals.n)
    return {"p": force, "n": -force}, {}

@component(ports=("p", "n"), discipline=mechanical)
def ForceSource(signals, force=0.1):
    return {"p": -force, "n": force}, {}

mechanical_net = {
    "instances": {"GND": {"component": "ground"}, "D1": {"component": "damper"}, "F1": {"component": "force"}},
    "connections": {"D1,p": "F1,p", "D1,n": "GND,p1", "F1,n": "GND,p1"},
}
mechanics = compile_circuit(mechanical_net, {"ground": lambda: 0, "damper": Damper, "force": ForceSource},
                            backend="dense", tolerances=TolerancePolicy(atol={"force": 1e-12}))
y, ok = mechanics.dc_checked()
assert ok
assert np.isclose(y[mechanics.port_map["D1,p"]], 0.05, atol=1e-8)
print("Velocity [m/s]:", float(y[mechanics.port_map["D1,p"]]))


## Validation and limits

Connecting a thermal port directly to an electrical port fails compilation in physical mode. Couple them through a component with explicit per-port disciplines, as above. Units are declarations, not automatic conversion. `Nature.nominal` optionally supplies a fixed relative residual scale; otherwise residual checks are absolute-only. OSDI and generated rational/SAX models require future metadata adapters before using this mode.